## 0. Kernel setup (run in a terminal, not in this notebook)

This notebook reuses the same `2ndWorkshop` conda environment and kernel as
`Workshop2_Part2a_Server.ipynb` and `Workshop2_Part2b_Agent.ipynb` -- if you've already
set that up and run the install cell in one of those notebooks, you can select the same
`Python (2ndWorkshop)` kernel here and skip straight to C1.

**Prerequisite:** `Workshop2_Part2a_Server.ipynb` must already be running (in a separate
kernel) before you run the cells below -- this notebook connects to the same MCP server
as Part 2b.

# Workshop 2, Part 2c (optional): Flexible ReAct Agent

**This notebook is an optional side-by-side comparison, not a required step.** Part 2b
settled on a *fixed-order workflow* graph (`gather_context` → `write`) where the model
physically cannot call `create_notification` before the lookup tools have run -- the
order is enforced by the graph's structure, not by asking nicely.

This notebook builds the *other* version: a single generic `agent` node, bound to every
tool at once, looping with a `ToolNode` until the model stops requesting tools -- the
classic **ReAct** pattern. Here, the *only* thing telling the model to look things up
before writing a notification is the system prompt (C5 below). Nothing in the code
prevents it from calling `create_notification` on its very first turn.

**Why keep this around?** It's a fair question whether the Part 2b rewrite was really
necessary, or whether a good-enough prompt (and a strong enough model) would have been
enough on its own. This notebook lets you test that directly: run the same questions
here and in Part 2b, across different `LLM_MODEL` presets, and compare whether the
placeholder-date bug (`[insert date]` written to `announcements.txt`) still shows up.
Expect the answer to depend on model quality -- prompting is a *soft* constraint the
model can still ignore, where Part 2b's graph shape is a *hard* one.

**Note:** this notebook writes to the same `workshop_outputs/announcements.txt` file as
Part 2b. Clear it between runs if you want a clean read on which version produced what.

## 0. Install dependencies

Run once, then restart the kernel. Same packages as Part 2b -- skip this if you already
ran it there in the same environment.

In [ ]:
! pip install torch transformers accelerate langchain-huggingface langchain langchain-core langgraph langchain-mcp-adapters langchain-ollama ollama python-dotenv


## C1. Imports for the agent

In [11]:
import os
import asyncio
import logging
from pathlib import Path
from typing import Annotated
from typing_extensions import TypedDict

from dotenv import load_dotenv
from transformers import pipeline
from langchain.chat_models import init_chat_model
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

load_dotenv()

BASE_DIR = Path(".").resolve()
ANNOUNCEMENTS_FILE = BASE_DIR / "workshop_outputs/announcements.txt"   # shared with Part 2b

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


## C2. Configure the LLM backend

Same two presets as Part 2b's C2 (local HF default, or Ollama) -- see that notebook's C2
markdown for the full explanation, install steps, and the other backend options (Purdue
GenAI / Anthropic / OpenAI). Since prompt-only ordering is exactly what's being tested
here, trying this notebook with a stronger backend (Ollama, or one of Part 2b's optional
presets) is the most interesting comparison -- the weak local default is the case most
likely to fail regardless of which architecture it's running in.

**Gilbreth + Ollama:** if `ollama serve` in a terminal isn't reachable from this
notebook (a separate kernel won't pick up a `PATH`/`~/.bashrc` change made after it
started, or a server you only launched inside *Part 2b's* kernel), run **C2a** below --
it's the same in-notebook launcher as Part 2b's C2a, so this notebook doesn't depend on
Part 2b's kernel staying alive. If Part 2b's Ollama server is already running and you'd
rather reuse it than start a second one, skip C2a and just make sure `LLM_BASE_URL`
below matches whatever port that one is on.

## C2a. (Gilbreth only, optional) Launch Ollama from inside this notebook

Skip this cell if `ollama serve` already works alongside this notebook (via a terminal,
or because Part 2b's C2a is running and this notebook's `LLM_BASE_URL` points at the
same port). Identical to Part 2b's C2a -- see that notebook for the full explanation.

In [ ]:
import subprocess
import time

OLLAMA_PORT = 11435   # keep in sync with LLM_BASE_URL in the preset cell below

# Same paths as Part 2b's C2a: models on scratch (avoids filling the small home quota),
# binary unpacked at ~/bin/bin/ollama.
os.environ["OLLAMA_MODELS"] = f"/scratch/gilbreth/{os.environ['USER']}/.ollama/models"
os.environ["OLLAMA_HOST"]   = f"127.0.0.1:{OLLAMA_PORT}"
ollama_executable = str(Path.home() / "bin" / "bin" / "ollama")   # adjust if you installed elsewhere

log_file = open("ollama_debug.log", "w")
print(f"Spawning Ollama server from: {ollama_executable}")

# Launched as a subprocess of *this* kernel, not a separate terminal -- so it inherits
# this process's environment directly instead of depending on a shell PATH/rc file that
# could be out of sync with what the kernel saw at startup.
ollama_process = subprocess.Popen(
    [ollama_executable, "serve"],
    stdout=log_file,
    stderr=log_file,
    env=os.environ,
)
time.sleep(4)   # give the server a moment to bind its port before the connection check below
print(f"Ollama server running under PID: {ollama_process.pid}")

# Sanity check -- confirms the notebook can actually reach the server before C2/C3 try to use it
from ollama import Client
client = Client(host=f"http://127.0.0.1:{OLLAMA_PORT}")
try:
    print("Connected. Loaded models:", client.list())
except Exception as e:
    print(f"Connection failed: {e}")
    print("Check ollama_debug.log -- often a port conflict or a wrong ollama_executable path.")


In [ ]:
# --- Edit these directly, no env vars needed. Uncomment ONE preset. ---

LOCAL_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # used only when LLM_MODEL is None (the local preset)

# 1) Local HF model (default) -- [local] runs anywhere (Mac or Gilbreth), no server or
#    API key, but weakest tool-calling reliability.
# LLM_MODEL    = None
# LLM_BASE_URL = None

# 2) Ollama -- [api, local server] best free tool-calling. See Part 2b's C2 for the
#    Gilbreth no-sudo install steps, or run C2a above. LLM_BASE_URL below assumes
#    C2a's port (11435); set it to None instead if you're running `ollama serve`
#    standalone on its default port (11434).
LLM_MODEL    = "ollama:llama3.2"
LLM_BASE_URL = "http://127.0.0.1:11435"


def create_llm():
    """
    Local Hugging Face model by default. Set LLM_MODEL above to a non-None value to
    route through init_chat_model() instead -- same mechanism as Part 2b.
    """
    if LLM_MODEL is None:
        text_gen = pipeline("text-generation", model=LOCAL_MODEL, max_new_tokens=512)
        return ChatHuggingFace(llm=HuggingFacePipeline(pipeline=text_gen))
    else:
        kwargs = {"temperature": 0}
        if LLM_BASE_URL:
            kwargs["base_url"] = LLM_BASE_URL
        return init_chat_model(LLM_MODEL, **kwargs)


print("LLM backend: ", f"local ({LOCAL_MODEL})" if LLM_MODEL is None else f"api ({LLM_MODEL})")


## C3. AgentState -- the shared message history

Back to the message-list shape (not Part 2b's plain string fields), because this is a
true multi-turn loop: every tool call and tool result needs to accumulate in one growing
history that the model re-reads on each turn. `add_messages` is the reducer that makes
new messages append instead of overwrite.

In [13]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


## C4. Build the LangGraph agent graph (generic ReAct loop)

```
START → agent ──┬─(tool_calls?)→ tools → agent (loop)
                └─(no tool calls)→ END
```

`agent_node` binds **every** discovered tool at once and invokes the model on the full
history; `should_continue` routes to `tools` whenever the last message has `tool_calls`,
otherwise ends. Nothing here restricts which tool the model can call on which turn --
the model is free to call `create_notification` first, `search_knowledge_base` first, or
skip a lookup entirely. Whether it does the right thing depends entirely on the system
prompt (C5) and the model's own reliability -- that's the whole point of this notebook.

In [14]:
def build_graph(tools):
    llm            = create_llm()
    # bind_tools attaches every tool's schema to every LLM call -- the model decides,
    # turn by turn, whether and which tool to call. No gating, no fixed order.
    llm_with_tools = llm.bind_tools(tools)

    def agent_node(state: AgentState) -> dict:
        response = llm_with_tools.invoke(state["messages"])
        return {"messages": [response]}

    def should_continue(state: AgentState) -> str:
        last_message = state["messages"][-1]
        if last_message.tool_calls:
            return "tools"
        return END

    tool_node = ToolNode(tools)

    graph = StateGraph(AgentState)
    graph.add_node("agent", agent_node)
    graph.add_node("tools", tool_node)
    graph.add_edge(START, "agent")
    graph.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
    graph.add_edge("tools", "agent")   # after tool execution, go back to agent
    return graph.compile()


## C5. System prompt -- this is the only thing enforcing order

This is Part 2b's most fully-worked-out prompt attempt (explicit ordering rules, an
explicit ban on placeholder text, and an explicit "re-call it if you already wrote it
wrong" instruction) -- a genuine best effort at solving this with prompting alone, not a
strawman. Edit it and re-run C5 + C6/C7 to try your own wording.

In [ ]:
SYSTEM_PROMPT = """
You are the Boilermaker Autonomous TA for a Purdue course.
Your job is to help students and instructors manage course planning using the tools available.

Assume today's date is September 1, 2026 -- the course is partway through its Fall
semester. Use this as "today" whenever the question is relative (e.g. "this week,"
"coming up," "next two weeks"), and when you propose a date for a new event, it must
fall within this Fall semester (not the following Spring) and must not be in the past.

Tools:
- search_knowledge_base(query): search the course knowledge base for syllabus details, policies, and study guidance.
- get_academic_calendar(query): read academic calendar events.
- create_notification(subject, body): write a formatted announcement to the announcements file.

Always use the tools when you need facts from the knowledge base or calendar.

Follow this order strictly:
1. First, call search_knowledge_base and/or get_academic_calendar as many times as needed
   to gather every specific fact the announcement will require (dates, times, policies).
   Do this before writing anything.
2. If the question asks about an existing fact (a policy, an exam date, a deadline) and
   the tool results don't contain it, say so plainly instead of guessing.
3. Some questions instead ask you to schedule a NEW event (e.g. a review session) that
   won't appear in any tool result, because it doesn't exist yet -- you have to invent
   it. In that case, pick a specific date and time yourself, using the retrieved
   academic calendar as constraints (avoid any date/time that conflicts with a listed
   holiday, exam, or deadline), and state your choice plainly as your own proposal.
4. Only after you have those results, call create_notification with every detail filled
   in -- using retrieved facts where they exist, or your own concrete proposed
   date/time for a new event you were asked to schedule. Never write placeholder text
   like "[insert date]" or "[insert time]" -- always commit to a real, specific value
   rather than leaving a gap.
5. create_notification writes to the announcements file immediately when called, and the
   file only reflects your most recent call -- it is not updated automatically to match
   whatever you say afterward. So if you already called create_notification earlier in
   this conversation and its content contains a placeholder or a value you have since
   corrected, you must call create_notification again with the corrected text as your
   final tool call before finishing.
"""


## C6. Connect to the MCP server and run the agent

In [ ]:
DEFAULT_QUESTION = (
    "A professor wants to schedule a single CS course review session that does not conflict "
    "with holidays or exams. Summarize the plan and write a notification announcement for students."
)

MCP_SERVER_URL = "http://127.0.0.1:8001/mcp"


async def run_agent(question: str = DEFAULT_QUESTION):
    log.info("Connecting to MCP server at %s", MCP_SERVER_URL)

    try:
        client = MultiServerMCPClient({
            "boiler_ta": {
                "url": MCP_SERVER_URL,
                "transport": "streamable_http",
            },
        })
        tools = await client.get_tools()
    except Exception as e:
        log.error("Could not connect to MCP server: %s", e)
        return

    log.info("Loaded %d tools: %s", len(tools), [t.name for t in tools])

    agent  = build_graph(tools)
    result = await agent.ainvoke({
        "messages": [SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=question)],
    })

    for msg in result["messages"]:
        if isinstance(msg, AIMessage) and msg.content:
            print("\nAgent reply:\n", msg.content)


# Run the async agent inside the notebook
await run_agent()


## C7. Try a custom question

Run the same question here and in Part 2b, then compare C8's `announcements.txt` output
between the two notebooks (clear the file between runs for a clean comparison).

In [ ]:
await run_agent("What assignment deadlines are coming up in the next two weeks? Summarize them for students.")


## C8. Check the announcements file

In [18]:
if ANNOUNCEMENTS_FILE.exists():
    print(ANNOUNCEMENTS_FILE.read_text())
else:
    print("No announcements written yet.")


Subject: Upcoming CS Course Review Session
The CS department will be holding a review session on 11/16 at 2 PM. Please note that this session does not conflict with any holidays or exams. We look forward to seeing you there! The course uses GitHub Classroom, Piazza for questions, and a shared lecture notes repository. Instructors recommend starting project planning at least three weeks before each milestone.
---



---
## Comparing this to Part 2b

A few things worth trying, to turn this into an actual experiment rather than a vibe:

- **Run the exact same `DEFAULT_QUESTION` in both notebooks**, with the same `LLM_MODEL`
  preset, clearing `announcements.txt` before each run. Does this notebook ever call
  `create_notification` before a lookup tool? Does the file ever end up with a
  placeholder while the printed "Agent reply" text looks correct (the original bug)?
- **Sweep across backends** (local HF, Ollama, and -- if you have a key -- Anthropic or
  OpenAI from Part 2b's optional cell). The gap between this notebook and Part 2b should
  shrink as the model gets stronger, but note whether it ever fully closes.
- **Try editing the system prompt** (C5) to be even more explicit, or add a worked
  example of correct tool-call ordering. Better prompting *can* reduce the failure rate
  here -- that's real, not a strawman -- but it's still a probability, not a guarantee.
  Part 2b's fixed graph doesn't need the model to get the order right, because there
  is no turn where the wrong order is even possible.

The honest takeaway: this ReAct version is more flexible (it can handle questions that
need a different sequence of tools, or no notification at all, without any code change),
but its correctness depends on prompt quality and model capability. Part 2b trades away
some of that flexibility for a guarantee. Which one is "better" depends on how much the
task's tool order can vary -- see the last bullet in Part 2b's Extension ideas.